# Linked expression summaries

Select cells in a UMAP to compare their gene expression with all cells.
Drag a box to select. Double-click to clear. The other plots update as you drag.

The count appears inside the box when it fits. Set `show_count=False` to hide it.


In [ ]:
import scanpy as sc
import genome_spy.integrations.scanpy as gsc

pbmc = sc.datasets.pbmc68k_reduced()
genes = ["CD3D", "CD79A", "MS4A1", "NKG7", "LYZ", "FCER1A"]

## Compare detected genes and marker expression

Select cells to compare their detected-gene counts and average gene expression
by cell-cycle phase.


In [ ]:
phase_qc = (
    gsc.pl.EmbeddingSummary(
        pbmc,
        genes,
        "phase",
        use_raw=False,
        obs_keys=["n_genes"],
        title="Cell-cycle phase",
        show_count=True,
    )
    .add_histogram(
        obs="n_genes",
        position="top",
        width=500,
        height=110,
        bin_width=100,
        title="Detected genes per cell",
    )
    .add_heatmap(
        ["CD3D", "NKG7", "LYZ"],
        position="right",
        width=180,
        height=160,
        cmap="viridis",
        vmin=-2.5,
        vmax=2.5,
        title="Selected marker means",
        show_values=True,
    )
    .style(umap_size=260, size=12, alpha=0.85)
    .legend(position="bottom")
    .to_chart()
)
phase_qc

## Compare gene expression

Gray bars show all cells; blue bars show your selection.


In [ ]:
marker_distribution = (
    gsc.pl.EmbeddingSummary(
        pbmc, genes, "bulk_labels", use_raw=False, title="Cell types", show_count=True
    )
    .add_histogram(
        ["LYZ", "CD3D"],
        position="left",
        width=150,
        height=75,
        bin_width=0.5,
        title=None,
    )
    .add_histogram(
        ["MS4A1", "NKG7", "FCER1A"],
        position="bottom",
        width=110,
        height=90,
        bin_width=0.5,
        title=None,
    )
    .style(umap_size=260, size=12, alpha=0.85)
    .legend(position="right")
    .to_chart()
)
marker_distribution

## Change the layout and colors

Use `position=` to place plots around the UMAP and `.legend(position=...)`
to move the legends. Set `show_values=True` to show numbers when they fit.

The `seurat` colors run from magenta through black at zero to yellow.


In [ ]:
plot = (
    gsc.pl.EmbeddingSummary(
        pbmc,
        genes,
        "bulk_labels",
        use_raw=False,
        obs_keys=["phase", "S_score"],
        title="PBMC cells",
        show_count=True,
    )
    .add_heatmap(
        position="bottom",
        width=660,
        height=160,
        cmap="seurat",
        vmin=-2.5,
        vmax=2.5,
        title="Selected marker expression",
        show_values=True,
    )
    .add_histogram(
        obs="S_score", position="right", width=150, bin_width=0.1, title="S-phase score"
    )
    .add_composition(
        "phase", position="right", width=130, height=240, title="Cell-cycle phase"
    )
    .style(umap_size=280, size=12, alpha=0.85, spacing=12)
    .legend(position="bottom")
)
linked_expression = plot.to_chart()
linked_expression

## Link UMAP to gene tracks

Scroll over the UMAP to zoom. Drag outside the selection to pan. Hold Shift and drag to select cells;
double-click to clear. Their bars stay bright in the tracks below.
Scroll over the tracks to zoom across cells and drag to pan. Hover for cell details.


In [ ]:
pbmc3k = sc.datasets.pbmc3k_processed()

In [ ]:
track_markers = ["CD3D", "MS4A1", "LYZ", "NKG7", "FCER1A"]
linked_tracks = (
    gsc.pl.EmbeddingSummary(
        pbmc3k,
        track_markers,
        "louvain",
        use_raw=True,
        title="PBMC3k cell types",
        show_count=True,
        zoom=True,
    )
    .add_tracks(
        width=480,
        height=200,
        zoom=True,
        group_gap=2,
        spacing=5,
        title="Marker expression",
    )
    .style(umap_size=480, size=6, alpha=0.85)
    .legend(position="bottom")
    .to_chart()
)
linked_tracks

:::{admonition} Data use and provenance
:class: note

The first three examples use 700 cells from Scanpy’s processed PBMC68k data.
The tracks use 2,638 cells from processed PBMC3k. Both come from 10x Genomics,
with processing supplied by Scanpy ([CC BY 4.0](https://creativecommons.org/licenses/by/4.0/)).
Existing UMAP coordinates and expression values are kept.
:::


## Read the summaries

Histograms show percentages within all cells or within your selection.
Histograms and averages skip missing values. Blank heatmap squares mean no
selected values are available. Everything updates in the browser, without a
running notebook.

Reduce the chosen cells, genes or cell details if you reach the 250,000-value limit.
Use Scanpy for statistical tests of expression differences.

:::{note}
Cells exactly on the top or right edge of a selection can appear in the plots
but be missing from the count and Python results.
:::
